In [6]:
import pandas as pd
import os
from folktables import ACSDataSource, ACSIncome
import numpy as np
import random
import torch

In [2]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [4]:
def load_state_data(state, year, sample_size=1000, task=ACSIncome):
    data_source = ACSDataSource(survey_year=year, horizon='1-Year', survey='person')
    state_data = data_source.get_data(states=[state], download=True)
    
    features, labels, _ = task.df_to_numpy(state_data)
    df = pd.DataFrame(features, columns=task.features)
    df["label"] = labels
    df["PINCP"] = state_data["PINCP"]
    
    df = df.drop_duplicates()
    
    df_sampled = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)
    
    return df_sampled

In [5]:
def create_sample_datasets(states, years, sample_sizes=[1000, 2000, 3000]):
    """Create multiple sampled datasets of different sizes."""
    for sample_size in sample_sizes:
        combined_data = []
        
        for year in years:
            for state in states:
                print(f"\nLoading data for year {year}, state {state} with sample size {sample_size}...")
                df_sampled = load_state_data(state, year, sample_size=sample_size)
                
                df_sampled['Year'] = year
                df_sampled['State'] = state
                
                combined_data.append(df_sampled)
        
        final_df = pd.concat(combined_data, ignore_index=True)
        
        output_dir = "sampled_datasets"
        os.makedirs(output_dir, exist_ok=True)
        
        output_file = os.path.join(output_dir, f"{sample_size}_samples.csv")
        final_df.to_csv(output_file, index=False)
        print(f"Saved {output_file} with {len(final_df)} total rows")
        print(f"Dataset shape: {final_df.shape}")
        print(f"Memory usage: {final_df.memory_usage().sum() / 1024 / 1024:.2f} MB")

In [6]:
def main():
    # Define states and years
    states = ["CA", "TX", "MI"]
    years = [2014, 2016, 2018]
    
    print("Starting dataset creation...")
    print(f"States: {states}")
    print(f"Years: {years}")
    
    # Create the sampled datasets
    create_sample_datasets(states, years)
    
    print("\nDataset creation completed!")

In [7]:
if __name__ == "__main__":
    main()

Starting dataset creation...
States: ['CA', 'TX', 'MI']
Years: [2014, 2016, 2018]

Loading data for year 2014, state CA with sample size 1000...

Loading data for year 2014, state TX with sample size 1000...

Loading data for year 2014, state MI with sample size 1000...

Loading data for year 2016, state CA with sample size 1000...

Loading data for year 2016, state TX with sample size 1000...

Loading data for year 2016, state MI with sample size 1000...

Loading data for year 2018, state CA with sample size 1000...


In [7]:
df = pd.read_csv('llm_income_prediction_analysis_1000.csv')
df.head()

,Year,State,Model,Response,Actual_Label,Gender,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P,PINCP
0,2014,CA,ChatGPT,Below,Below,Male,30.0,1.0,22.0,1.0,4850.0,515.0,0.0,40.0,1.0,1.0,1200.0
1,2014,CA,Claude,Below,Below,Male,30.0,1.0,22.0,1.0,4850.0,515.0,0.0,40.0,1.0,1.0,1200.0
2,2014,CA,Mistral,Above,Below,Male,30.0,1.0,22.0,1.0,4850.0,515.0,0.0,40.0,1.0,1.0,1200.0
3,2014,CA,ChatGPT,Below,Below,Female,40.0,4.0,22.0,5.0,2200.0,6.0,2.0,10.0,2.0,1.0,NaN
4,2014,CA,Claude,Below,Below,Female,40.0,4.0,22.0,5.0,2200.0,6.0,2.0,10.0,2.0,1.0,NaN


In [8]:
original_samples = df[df['Model'] == 'ChatGPT'].copy()
original_samples.head()


,Year,State,Model,Response,Actual_Label,Gender,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P,PINCP
0,2014,CA,ChatGPT,Below,Below,Male,30.0,1.0,22.0,1.0,4850.0,515.0,0.0,40.0,1.0,1.0,1200.0
3,2014,CA,ChatGPT,Below,Below,Female,40.0,4.0,22.0,5.0,2200.0,6.0,2.0,10.0,2.0,1.0,NaN
6,2014,CA,ChatGPT,Below,Above,Male,44.0,4.0,22.0,1.0,2310.0,33.0,0.0,50.0,1.0,1.0,38400.0
9,2014,CA,ChatGPT,Above,Above,Male,62.0,6.0,21.0,1.0,430.0,36.0,0.0,60.0,1.0,1.0,5000.0
12,2014,CA,ChatGPT,Below,Below,Female,57.0,1.0,16.0,4.0,4020.0,212.0,0.0,40.0,2.0,1.0,NaN


In [9]:
original_samples.shape

(9000, 17)

In [10]:
original_samples = original_samples.reset_index(drop=True)

In [11]:
print(f"Number of unique samples: {len(original_samples)}")

Number of unique samples: 9000


In [13]:
original_samples = original_samples.drop('Model', axis=1)

In [15]:
original_samples = original_samples.drop('Response', axis=1)

In [16]:
original_samples.head()

,Year,State,Actual_Label,Gender,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P,PINCP
0,2014,CA,Below,Male,30.0,1.0,22.0,1.0,4850.0,515.0,0.0,40.0,1.0,1.0,1200.0
1,2014,CA,Below,Female,40.0,4.0,22.0,5.0,2200.0,6.0,2.0,10.0,2.0,1.0,NaN
2,2014,CA,Above,Male,44.0,4.0,22.0,1.0,2310.0,33.0,0.0,50.0,1.0,1.0,38400.0
3,2014,CA,Above,Male,62.0,6.0,21.0,1.0,430.0,36.0,0.0,60.0,1.0,1.0,5000.0
4,2014,CA,Below,Female,57.0,1.0,16.0,4.0,4020.0,212.0,0.0,40.0,2.0,1.0,NaN


In [17]:
original_samples.to_csv('original_1000_samples.csv', index=False)